# DR2 · TP2 — Estruturas de Dados e Algoritmos

**Aluno(a):** _(preencher)_

Cada exercício tem uma nota curta (decisão e Big O), o código e a execução. O mesmo código está em `python/exNN_*.py` e roda com `python python/ex01_maior_unico.py`.

*Comparação* = comparação entre elementos. *Acesso* = chamada a `put`/`get`/`contains` da hash table.

## Hash table própria (usada nos Ex. 1, 2, 3 e 5)

Encadeamento separado, capacidade prima e redimensionamento com fator de carga ≤ 0,75. Isso mantém `put`/`get`/`contains` em `O(1)` amortizado, e o contador `acessos` mede o custo.

In [1]:
def _proximo_primo(n):
    while True:
        if n >= 2 and all(n % d for d in range(2, int(n ** 0.5) + 1)):
            return n
        n += 1


class HashTable:
    def __init__(self, capacidade=17, fator_max=0.75):
        self.capacidade = capacidade
        self.fator_max = fator_max
        self.buckets = [[] for _ in range(capacidade)]
        self.tamanho = 0
        self.acessos = 0                       # nº de put/get/contains

    @staticmethod
    def _hash(chave):
        if isinstance(chave, str):             # hash polinomial para strings
            h = 0
            for ch in chave:
                h = (h * 31 + ord(ch)) & 0xFFFFFFFF
            return h
        return chave                           # inteiros: o próprio valor

    def _bucket(self, chave):
        return self.buckets[self._hash(chave) % self.capacidade]

    def put(self, chave, valor):
        self.acessos += 1
        bucket = self._bucket(chave)
        for par in bucket:
            if par[0] == chave:
                par[1] = valor
                return
        bucket.append([chave, valor])
        self.tamanho += 1
        if self.tamanho / self.capacidade > self.fator_max:
            self._redimensionar()

    def get(self, chave, padrao=None):
        self.acessos += 1
        for k, v in self._bucket(chave):
            if k == chave:
                return v
        return padrao

    def contains(self, chave):
        self.acessos += 1
        return any(k == chave for k, _ in self._bucket(chave))

    def _redimensionar(self):                  # custo amortizado O(1) por inserção
        antigos = self.buckets
        self.capacidade = _proximo_primo(self.capacidade * 2)
        self.buckets = [[] for _ in range(self.capacidade)]
        for bucket in antigos:
            for k, v in bucket:
                self.buckets[self._hash(k) % self.capacidade].append([k, v])

In [2]:
t = HashTable()
for i in range(1000):
    t.put(i * 16, i)
assert t.tamanho == 1000 and all(t.get(i * 16) == i for i in range(1000))
assert not t.contains(5) and t.contains(32)
print("HashTable OK | capacidade final:", t.capacidade)

HashTable OK | capacidade final: 1361


## Exercício 1 — Maior número único

**Gargalo:** para cada elemento, o laço interno relê a lista inteira atrás de um igual. No pior caso (valores distintos) são `N² − 1` comparações, `Θ(N²)`.
**Solução:** contar as frequências uma vez na hash table e depois consultar a frequência de cada elemento, `3N` acessos: `O(N)` tempo e `O(N)` espaço.
**Resultado:** ao dobrar `N`, as comparações quadruplicam e os acessos dobram. Os `assert` confirmam `N² − 1` e `3N` exatos.

In [3]:
import random


def maior_unico_quadratico(lista):
    """Retorna (resultado, comparações)."""
    comparacoes = 0
    melhor = None
    for i in range(len(lista)):
        unico = True
        for j in range(len(lista)):
            if i != j:
                comparacoes += 1
                if lista[i] == lista[j]:
                    unico = False
                    break
        if unico:
            if melhor is None:
                melhor = lista[i]
            else:
                comparacoes += 1
                if lista[i] > melhor:
                    melhor = lista[i]
    return melhor, comparacoes


def maior_unico_hash(lista):
    """Retorna (resultado, acessos à hash table)."""
    freq = HashTable()
    for x in lista:                            # 1ª passada: 2 acessos por elemento
        freq.put(x, freq.get(x, 0) + 1)
    melhor = None
    for x in lista:                            # 2ª passada: 1 acesso por elemento
        if freq.get(x) == 1 and (melhor is None or x > melhor):
            melhor = x
    return melhor, freq.acessos

In [4]:
for lista, esperado in [([4, 1, 2, 2, 4, 7, 7, 3], 3), ([5, 5], None), ([], None), ([9], 9)]:
    assert maior_unico_quadratico(lista)[0] == maior_unico_hash(lista)[0] == esperado
print("Casos pequenos OK\n")

random.seed(42)
print(f"{'N':>6} | {'comparações O(N²)':>18} | {'acessos hash':>12}")
for n in [100, 500, 1000, 2000, 5000]:
    lista = random.sample(range(10 * n), n)          # todos únicos = pior caso
    r1, comps = maior_unico_quadratico(lista)
    r2, acessos = maior_unico_hash(lista)
    assert r1 == r2 == max(lista)
    assert comps == n * n - 1 and acessos == 3 * n   # fórmulas teóricas
    print(f"{n:>6} | {comps:>18,} | {acessos:>12,}")

Casos pequenos OK

     N |  comparações O(N²) | acessos hash
   100 |              9,999 |          300
   500 |            249,999 |        1,500
  1000 |            999,999 |        3,000


  2000 |          3,999,999 |        6,000


  5000 |         24,999,999 |       15,000


## Exercício 2 — Interseção de arrays

**Gargalo:** varrer `A` para cada elemento de `B` é `O(N·M)`.
**Solução:** guardar `A` na hash table (`N` acessos) e percorrer `B` (até `2M` acessos, o segundo é a marcação que evita repetir no resultado): `O(N + M)`, sem funções prontas de interseção.
**Validação:** com `N = M = 200.000`, o resultado esperado é conhecido por construção (`[N/2, N)`). O quadrático faria ~2·10¹⁰ comparações.

In [5]:
import random


def interseccao_quadratica(a, b):
    """Linha de base O(N·M). Retorna (resultado, comparações)."""
    comps, resultado = 0, []
    for x in b:
        achou = False
        for y in a:
            comps += 1
            if x == y:
                achou = True
                break
        if achou:
            repetido = False
            for r in resultado:
                comps += 1
                if r == x:
                    repetido = True
                    break
            if not repetido:
                resultado.append(x)
    return resultado, comps


def interseccao_hash(a, b):
    """Retorna (resultado, acessos). Sem funções prontas de interseção."""
    tabela = HashTable()
    for x in a:
        tabela.put(x, True)                    # N acessos
    resultado = []
    for y in b:
        if tabela.get(y) is True:              # M acessos
            resultado.append(y)
            tabela.put(y, False)               # marca: não repete no resultado
    return resultado, tabela.acessos

In [6]:
for a, b, esperado in [([1, 2, 3, 4], [3, 4, 5, 6], [3, 4]),
                       ([1, 2, 2, 3], [2, 2, 3, 3, 9], [2, 3]),
                       ([], [1], []), (["a", "b"], ["b", "z"], ["b"])]:
    assert interseccao_quadratica(a, b)[0] == interseccao_hash(a, b)[0] == esperado
print("Casos pequenos OK\n")

print(f"{'N = M':>6} | {'comparações O(N·M)':>19} | {'acessos hash':>12}")
for n in [200, 1000, 2000]:
    a, b = list(range(n)), list(range(n // 2, n // 2 + n))
    random.shuffle(b)
    (r1, comps), (r2, acessos) = interseccao_quadratica(a, b), interseccao_hash(a, b)
    assert r1 == r2 and len(r2) == n // 2 and acessos <= 3 * n
    print(f"{n:>6} | {comps:>19,} | {acessos:>12,}")

# Validação em escala: resposta conhecida por construção ([N/2, N)), sem usar set
N = 200_000
a, b = list(range(N)), list(range(N // 2, N // 2 + N))
random.shuffle(a); random.shuffle(b)
resultado, acessos = interseccao_hash(a, b)
marcado = [False] * N
for r in resultado:
    assert N // 2 <= r < N and not marcado[r]
    marcado[r] = True
assert len(resultado) == N // 2
print(f"\nN = M = {N:,}: interseção com {len(resultado):,} elementos, {acessos:,} acessos (≤ 3N). OK")

Casos pequenos OK

 N = M |  comparações O(N·M) | acessos hash
   200 |              40,000 |          500
  1000 |           1,000,000 |        2,500
  2000 |           4,000,000 |        5,000



N = M = 200,000: interseção com 100,000 elementos, 500,000 acessos (≤ 3N). OK


## Exercício 3 — Primeiro valor duplicado

Uma passada: se a string já está na hash table, é o duplicado e a função retorna na hora. Senão, é inserida. São no máximo 2 acessos por elemento, `2N + 1` no pior caso: **`O(N)`**. Ordenar antes seria `O(N log N)` e perderia a ordem de leitura.

In [7]:
def primeiro_duplicado(arr):
    """Retorna (valor duplicado, acessos). Para na primeira repetição detectada."""
    vistos = HashTable()
    for s in arr:
        if vistos.contains(s):
            return s, vistos.acessos
        vistos.put(s, True)
    return None, vistos.acessos

In [8]:
testes = [(["maçã", "banana", "laranja", "uva", "banana", "pera"], "banana"),
          (["x", "y", "y"], "y"),
          (["a", "b", "c", "d", "a"], "a"),
          (["Python", "python", "Java", "C", "Java"], "Java"),   # diferencia maiúsculas
          (["", "a", ""], "")]
for arr, esperado in testes:
    valor, acessos = primeiro_duplicado(arr)
    assert valor == esperado
    print(f"{str(arr):58} -> {valor!r:9} ({acessos} acessos)")

print()
for n in [1_000, 10_000, 100_000]:
    arr = [f"id{i}" for i in range(n)] + ["id0"]    # duplicata só no fim = pior caso
    valor, acessos = primeiro_duplicado(arr)
    assert valor == "id0" and acessos == 2 * n + 1
    print(f"N = {n + 1:>7,}: {acessos:>8,} acessos (= 2N + 1)")

['maçã', 'banana', 'laranja', 'uva', 'banana', 'pera']     -> 'banana'  (9 acessos)
['x', 'y', 'y']                                            -> 'y'       (5 acessos)
['a', 'b', 'c', 'd', 'a']                                  -> 'a'       (9 acessos)
['Python', 'python', 'Java', 'C', 'Java']                  -> 'Java'    (9 acessos)
['', 'a', '']                                              -> ''        (5 acessos)

N =   1,001:    2,001 acessos (= 2N + 1)
N =  10,001:   20,001 acessos (= 2N + 1)


N = 100,001:  200,001 acessos (= 2N + 1)


## Exercício 4 — Letra ausente do alfabeto

Um vetor de 26 posições marca a presença de cada letra (`índice = ord(c) − ord('a')`). Maiúsculas são normalizadas e tudo que não é `a–z` é ignorado.
**Big O:** tempo `O(N + 26) = O(N)`; espaço `O(1)`, pois o vetor tem tamanho fixo.

In [9]:
def letras_ausentes(texto):
    presente = [False] * 26                    # estrutura auxiliar de tamanho fixo
    for ch in texto:                           # N iterações
        if "A" <= ch <= "Z":
            ch = chr(ord(ch) + 32)             # normaliza para minúscula
        if "a" <= ch <= "z":                   # ignora espaços, números, pontuação, acentos
            presente[ord(ch) - ord("a")] = True
    return [chr(ord("a") + i) for i in range(26) if not presente[i]]   # 26 iterações

In [10]:
testes = [("The quick brown fox jumps over the lazy do", ["g"]),
          ("The quick brown fox jumps over the lazy dog", []),        # pangrama
          ("Pack my box with five dozen liquor jug!!! 123", ["s"]),
          ("  ABCDEFGHIJKLMNOPQRSTUVWXY  ", ["z"])]
for texto, esperado in testes:
    r = letras_ausentes(texto)
    assert r == esperado
    print(f"{texto!r:50} -> ausentes: {r}")

'The quick brown fox jumps over the lazy do'       -> ausentes: ['g']
'The quick brown fox jumps over the lazy dog'      -> ausentes: []
'Pack my box with five dozen liquor jug!!! 123'    -> ausentes: ['s']
'  ABCDEFGHIJKLMNOPQRSTUVWXY  '                    -> ausentes: ['z']


## Exercício 5 — Primeiro caractere não repetido

Duas fases: (1) contar a frequência de cada caractere na hash table; (2) percorrer a **string** na ordem original e devolver o primeiro com frequência 1. Isso preserva a ordem sem recontar. **`O(N)`**, `3N` acessos no pior caso.

In [11]:
def primeiro_nao_repetido(texto):
    """Retorna (caractere ou None, acessos)."""
    freq = HashTable()
    for ch in texto:                           # fase 1: contagem de frequência
        freq.put(ch, freq.get(ch, 0) + 1)
    for ch in texto:                           # fase 2: ordem original preservada
        if freq.get(ch) == 1:
            return ch, freq.acessos
    return None, freq.acessos

In [12]:
testes = [("swiss", "w"), ("stress", "t"), ("abcabcd", "d"), ("aabbcc", None),
          ("", None), ("x", "x"), ("Aa", "A"), ("hello world", "h")]
for texto, esperado in testes:
    r, acessos = primeiro_nao_repetido(texto)
    assert r == esperado
    print(f"{texto!r:14} -> {r!r:5} ({acessos} acessos)")

texto = "abcdefghij" * 50_000 + "Z"                       # resposta no fim = pior caso
r, acessos = primeiro_nao_repetido(texto)
assert r == "Z" and acessos == 3 * len(texto)
print(f"\nN = {len(texto):,}: {acessos:,} acessos (= 3N)")

'swiss'        -> 'w'   (12 acessos)
'stress'       -> 't'   (14 acessos)
'abcabcd'      -> 'd'   (21 acessos)
'aabbcc'       -> None  (18 acessos)
''             -> None  (0 acessos)
'x'            -> 'x'   (3 acessos)
'Aa'           -> 'A'   (5 acessos)
'hello world'  -> 'h'   (23 acessos)



N = 500,001: 1,500,003 acessos (= 3N)


## Exercício 6 — Selection Sort

A cada passo, acha o menor do trecho não ordenado e o troca para a posição `i`.
**Resultado:** as comparações são **sempre `N(N−1)/2` (499.500 para N = 1000)**, em qualquer estado inicial. Só as trocas variam, no máximo `N−1`.
**Big O:** `O(N²)` no melhor, médio e pior caso; espaço `O(1)`; não é estável.

In [13]:
import random


def selection_sort(lista):
    """Retorna (lista ordenada, comparações, trocas)."""
    a = list(lista)
    comparacoes = trocas = 0
    for i in range(len(a) - 1):
        menor = i
        for j in range(i + 1, len(a)):
            comparacoes += 1
            if a[j] < a[menor]:
                menor = j
        if menor != i:
            a[i], a[menor] = a[menor], a[i]
            trocas += 1
    return a, comparacoes, trocas


def gerar_cenarios(n, seed=7):
    """Cinco estados iniciais para o mesmo N."""
    rng = random.Random(seed)
    quase = list(range(n))
    for _ in range(10):                        # 10 trocas curtas
        i = rng.randrange(n - 3)
        j = i + rng.randint(1, 3)
        quase[i], quase[j] = quase[j], quase[i]
    return {"ordenada": list(range(n)), "quase ordenada": quase,
            "aleatória": rng.sample(range(10 * n), n), "invertida": list(range(n, 0, -1)),
            "muitos duplicados": [rng.randint(0, 9) for _ in range(n)]}

In [14]:
N = 1000
print(f"{'estado inicial':>18} | {'comparações':>11} | {'trocas':>6}")
for nome, lista in gerar_cenarios(N).items():
    saida, comps, trocas = selection_sort(lista)
    assert saida == sorted(lista)                        # sorted() só para conferir
    assert comps == N * (N - 1) // 2 and trocas <= N - 1
    print(f"{nome:>18} | {comps:>11,} | {trocas:>6}")

    estado inicial | comparações | trocas
          ordenada |     499,500 |      0
    quase ordenada |     499,500 |     10
         aleatória |     499,500 |    993
         invertida |     499,500 |    500
 muitos duplicados |     499,500 |    896


## Exercício 7 — Insertion Sort

Insere cada elemento no prefixo já ordenado, deslocando os maiores para a direita.
**Resultado:** lista ordenada: 999 comparações e 0 deslocamentos. Quase ordenada: 1.025 comparações. Invertida: 499.500 comparações e 499.500 deslocamentos. O Selection Sort faz 499.500 comparações em todos os casos.
**Big O:** melhor caso `O(N)` (já ordenada), pior caso `O(N²)` (invertida), médio `O(N²)`. Espaço `O(1)`; estável.

In [15]:
def insertion_sort(lista):
    """Retorna (lista ordenada, comparações, deslocamentos)."""
    a = list(lista)
    comparacoes = deslocamentos = 0
    for i in range(1, len(a)):
        chave = a[i]
        j = i - 1
        while j >= 0:
            comparacoes += 1
            if a[j] > chave:
                a[j + 1] = a[j]                # desloca para a direita
                deslocamentos += 1
                j -= 1
            else:
                break
        a[j + 1] = chave
    return a, comparacoes, deslocamentos

In [16]:
N = 1000
print(f"{'estado inicial':>18} | {'Insertion: comps':>16} | {'deslocs':>8} | {'Selection: comps':>16}")
resultados = {}
for nome, lista in gerar_cenarios(N).items():
    saida, comps, desl = insertion_sort(lista)
    assert saida == sorted(lista)
    resultados[nome] = (comps, desl)
    print(f"{nome:>18} | {comps:>16,} | {desl:>8,} | {selection_sort(lista)[1]:>16,}")

assert resultados["ordenada"] == (N - 1, 0)                                  # melhor caso
assert resultados["invertida"] == (N * (N - 1) // 2, N * (N - 1) // 2)       # pior caso
print("\nMelhor caso (ordenada): N-1 comparações, 0 deslocamentos.")
print("Pior caso (invertida): N(N-1)/2 comparações e deslocamentos.")

    estado inicial | Insertion: comps |  deslocs | Selection: comps
          ordenada |              999 |        0 |          499,500
    quase ordenada |            1,025 |       26 |          499,500
         aleatória |          258,382 |  257,393 |          499,500


         invertida |          499,500 |  499,500 |          499,500


 muitos duplicados |          221,366 |  220,369 |          499,500

Melhor caso (ordenada): N-1 comparações, 0 deslocamentos.
Pior caso (invertida): N(N-1)/2 comparações e deslocamentos.


## Exercício 8 — Pilha ou fila?

**Escolha: fila (FIFO).**

- A regra "atender estritamente na ordem de chegada" é a definição de FIFO. Com pilha (LIFO) a solicitação mais recente seria atendida primeiro e as antigas ficariam esperando indefinidamente.
- Com array circular e dois índices, todas as operações são `O(1)`. `lista.pop(0)` custaria `O(N)` por remoção.
- Capacidade fixa permite tratar overflow (sistema saturado) e underflow (nada a atender) de forma explícita.

| Operação | Efeito | Custo |
|---|---|---|
| `enqueue(x)` | insere no fim (chegou uma solicitação) | `O(1)` |
| `dequeue()` | remove do início (atendente livre) | `O(1)` |
| `esta_vazia()` / `esta_cheia()` / `tamanho()` | consulta de estado | `O(1)` |

A demonstração executável está logo após o Exercício 10.

## Exercício 9 — Pilha (`Stack`) com overflow e underflow

**Persistência de estado:** o array e o índice do topo são atributos da instância e mudam in-place a cada operação. O mesmo objeto é usado do começo ao fim, e o estado impresso após cada passo comprova isso.
`push` em pilha cheia lança `StackOverflowError`; `pop` em pilha vazia lança `StackUnderflowError`. Em ambos os casos o estado **não muda** (conferido por `assert`). Todas as operações são `O(1)`.

In [17]:
class StackOverflowError(Exception):
    pass


class StackUnderflowError(Exception):
    pass


class Stack:
    def __init__(self, capacidade):
        self._capacidade = capacidade
        self._dados = [None] * capacidade
        self._topo = -1                        # -1 = pilha vazia

    def tamanho(self):
        return self._topo + 1

    def esta_vazia(self):
        return self._topo == -1

    def esta_cheia(self):
        return self._topo == self._capacidade - 1

    def push(self, item):
        if self.esta_cheia():
            raise StackOverflowError(f"pilha cheia; {item!r} não inserido")
        self._topo += 1
        self._dados[self._topo] = item

    def pop(self):
        if self.esta_vazia():
            raise StackUnderflowError("pilha vazia; nada a remover")
        item = self._dados[self._topo]
        self._dados[self._topo] = None
        self._topo -= 1
        return item

    def peek(self):
        if self.esta_vazia():
            raise StackUnderflowError("pilha vazia; sem topo")
        return self._dados[self._topo]

    def estado(self):
        return f"dados={self._dados} topo={self._topo} tamanho={self.tamanho()}/{self._capacidade}"

In [18]:
pilha = Stack(3)
print("criada:", pilha.estado())
operacoes = [("pop()", pilha.pop), ("push(10)", lambda: pilha.push(10)),
             ("push(20)", lambda: pilha.push(20)), ("push(30)", lambda: pilha.push(30)),
             ("push(40)", lambda: pilha.push(40)), ("peek()", pilha.peek),
             ("pop()", pilha.pop), ("pop()", pilha.pop), ("pop()", pilha.pop), ("pop()", pilha.pop)]
for nome, op in operacoes:
    antes = pilha.estado()
    try:
        r = op()
        print(f"{nome:10} ok{'' if r is None else f' -> {r}'}")
    except (StackOverflowError, StackUnderflowError) as e:
        print(f"{nome:10} ERRO {type(e).__name__}: {e}")
        assert pilha.estado() == antes     # erro não altera o estado
    print("           ", pilha.estado())

criada: dados=[None, None, None] topo=-1 tamanho=0/3
pop()      ERRO StackUnderflowError: pilha vazia; nada a remover
            dados=[None, None, None] topo=-1 tamanho=0/3
push(10)   ok
            dados=[10, None, None] topo=0 tamanho=1/3
push(20)   ok
            dados=[10, 20, None] topo=1 tamanho=2/3
push(30)   ok
            dados=[10, 20, 30] topo=2 tamanho=3/3
push(40)   ERRO StackOverflowError: pilha cheia; 40 não inserido
            dados=[10, 20, 30] topo=2 tamanho=3/3
peek()     ok -> 30
            dados=[10, 20, 30] topo=2 tamanho=3/3
pop()      ok -> 30
            dados=[10, 20, None] topo=1 tamanho=2/3
pop()      ok -> 20
            dados=[10, None, None] topo=0 tamanho=1/3
pop()      ok -> 10
            dados=[None, None, None] topo=-1 tamanho=0/3
pop()      ERRO StackUnderflowError: pilha vazia; nada a remover
            dados=[None, None, None] topo=-1 tamanho=0/3


## Exercício 10 — Fila (`Queue`) com ponteiros

Array circular com ponteiro de **início** (próximo a sair), ponteiro de **fim** (próxima posição livre) e contador de tamanho. Os ponteiros avançam com `(p + 1) mod capacidade`, então posições liberadas são reaproveitadas, como mostra o `fim` voltando ao índice 0 na execução. O contador distingue fila cheia de fila vazia, já que nos dois casos `inicio == fim`.
Overflow e underflow lançam exceções próprias sem alterar o estado. **FIFO** validado: `saíram == entraram`. Operações `O(1)`.

In [19]:
class QueueOverflowError(Exception):
    pass


class QueueUnderflowError(Exception):
    pass


class Queue:
    def __init__(self, capacidade):
        self._capacidade = capacidade
        self._dados = [None] * capacidade
        self._inicio = 0                       # próximo a sair
        self._fim = 0                          # próxima posição livre
        self._tamanho = 0

    def tamanho(self):
        return self._tamanho

    def esta_vazia(self):
        return self._tamanho == 0

    def esta_cheia(self):
        return self._tamanho == self._capacidade

    def enqueue(self, item):
        if self.esta_cheia():
            raise QueueOverflowError(f"fila cheia; {item!r} não inserido")
        self._dados[self._fim] = item
        self._fim = (self._fim + 1) % self._capacidade
        self._tamanho += 1

    def dequeue(self):
        if self.esta_vazia():
            raise QueueUnderflowError("fila vazia; nada a remover")
        item = self._dados[self._inicio]
        self._dados[self._inicio] = None
        self._inicio = (self._inicio + 1) % self._capacidade
        self._tamanho -= 1
        return item

    def estado(self):
        return (f"dados={self._dados} inicio={self._inicio} fim={self._fim} "
                f"tamanho={self._tamanho}/{self._capacidade}")

In [20]:
fila = Queue(4)
entraram, sairam = [], []
print("criada:", fila.estado())
roteiro = ["deq", "A", "B", "C", "D", "E", "deq", "deq", "F", "G", "deq", "deq", "deq", "deq", "deq"]
for passo in roteiro:
    antes = fila.estado()
    try:
        if passo == "deq":
            x = fila.dequeue()
            sairam.append(x)
            print(f"dequeue() -> {x!r}")
        else:
            fila.enqueue(passo)
            entraram.append(passo)
            print(f"enqueue({passo!r})")
    except (QueueOverflowError, QueueUnderflowError) as e:
        print(f"{'dequeue()' if passo == 'deq' else f'enqueue({passo!r})'} ERRO {type(e).__name__}: {e}")
        assert fila.estado() == antes
    print("           ", fila.estado())

print("\nEntraram:", entraram)
print("Saíram:  ", sairam)
assert entraram == sairam
print("FIFO validado: ordem de saída = ordem de entrada.")

criada: dados=[None, None, None, None] inicio=0 fim=0 tamanho=0/4
dequeue() ERRO QueueUnderflowError: fila vazia; nada a remover
            dados=[None, None, None, None] inicio=0 fim=0 tamanho=0/4
enqueue('A')
            dados=['A', None, None, None] inicio=0 fim=1 tamanho=1/4
enqueue('B')
            dados=['A', 'B', None, None] inicio=0 fim=2 tamanho=2/4
enqueue('C')
            dados=['A', 'B', 'C', None] inicio=0 fim=3 tamanho=3/4
enqueue('D')
            dados=['A', 'B', 'C', 'D'] inicio=0 fim=0 tamanho=4/4
enqueue('E') ERRO QueueOverflowError: fila cheia; 'E' não inserido
            dados=['A', 'B', 'C', 'D'] inicio=0 fim=0 tamanho=4/4
dequeue() -> 'A'
            dados=[None, 'B', 'C', 'D'] inicio=1 fim=0 tamanho=3/4
dequeue() -> 'B'
            dados=[None, None, 'C', 'D'] inicio=2 fim=0 tamanho=2/4
enqueue('F')
            dados=['F', None, 'C', 'D'] inicio=2 fim=1 tamanho=3/4
enqueue('G')
            dados=['F', 'G', 'C', 'D'] inicio=2 fim=2 tamanho=4/4
dequeue() -> 'C'
 

### Exercício 8 — demonstração: fila × pilha

As mesmas cinco solicitações processadas pelas duas estruturas. Só a fila respeita a ordem de chegada.

In [21]:
chegadas = ["R1", "R2", "R3", "R4", "R5"]
fila, pilha = Queue(len(chegadas)), Stack(len(chegadas))
for r in chegadas:
    fila.enqueue(r)
    pilha.push(r)
ordem_fila = [fila.dequeue() for _ in chegadas]
ordem_pilha = [pilha.pop() for _ in chegadas]
print("Chegada:        ", chegadas)
print("Atendido (FILA):", ordem_fila)
print("Atendido (PILHA):", ordem_pilha)
assert ordem_fila == chegadas and ordem_pilha != chegadas

Chegada:         ['R1', 'R2', 'R3', 'R4', 'R5']
Atendido (FILA): ['R1', 'R2', 'R3', 'R4', 'R5']
Atendido (PILHA): ['R5', 'R4', 'R3', 'R2', 'R1']


## Exercício 11 — Reversão de string com pilha

Cada caractere sofre um `push` explícito e depois um `pop` explícito, sem slicing nem `reversed`. A saída vem da ordem LIFO. **`O(N)`** tempo e espaço. O rastro de `'Pilha'` mostra o estado da pilha a cada passo.

In [22]:
def inverter_string(texto, verbose=False):
    pilha = Stack(max(1, len(texto)))
    for ch in texto:                           # empilha cada caractere
        pilha.push(ch)
        if verbose:
            print(f"  push({ch!r})  {pilha.estado()}")
    saida = []
    while not pilha.esta_vazia():              # desempilha até esvaziar
        ch = pilha.pop()
        saida.append(ch)
        if verbose:
            print(f"  pop() -> {ch!r}  {pilha.estado()}")
    return "".join(saida)

In [23]:
print("Rastro para 'Pilha':")
assert inverter_string("Pilha", verbose=True) == "ahliP"
print()
for s in ["Python", "", "a", "arara", "Estruturas de Dados", "ação é útil"]:
    r = inverter_string(s)
    n = len(s)
    assert len(r) == n and all(r[i] == s[n - 1 - i] for i in range(n))
    print(f"{s!r:24} -> {r!r}")

Rastro para 'Pilha':
  push('P')  dados=['P', None, None, None, None] topo=0 tamanho=1/5
  push('i')  dados=['P', 'i', None, None, None] topo=1 tamanho=2/5
  push('l')  dados=['P', 'i', 'l', None, None] topo=2 tamanho=3/5
  push('h')  dados=['P', 'i', 'l', 'h', None] topo=3 tamanho=4/5
  push('a')  dados=['P', 'i', 'l', 'h', 'a'] topo=4 tamanho=5/5
  pop() -> 'a'  dados=['P', 'i', 'l', 'h', None] topo=3 tamanho=4/5
  pop() -> 'h'  dados=['P', 'i', 'l', None, None] topo=2 tamanho=3/5
  pop() -> 'l'  dados=['P', 'i', None, None, None] topo=1 tamanho=2/5
  pop() -> 'i'  dados=['P', None, None, None, None] topo=0 tamanho=1/5
  pop() -> 'P'  dados=[None, None, None, None, None] topo=-1 tamanho=0/5

'Python'                 -> 'nohtyP'
''                       -> ''
'a'                      -> 'a'
'arara'                  -> 'arara'
'Estruturas de Dados'    -> 'sodaD ed saruturtsE'
'ação é útil'            -> 'litú é oãça'


## Exercício 12 — Funções hash (`M = 10`)

`h₁ = k mod 10`, `h₂ = 7k mod 10`, `h₃ = k² mod 10`. As três dependem **só do último dígito** da chave, então basta olhar os 10 resíduos.

- **`h₂` é equivalente a `h₁`.** Como `mdc(7, 10) = 1`, multiplicar por 7 só **permuta** os buckets: mesma carga em todos os conjuntos testados (`χ²` idêntico), sem ganho nenhum.
- **`h₃` é a pior.** Quadrados terminam só em `{0, 1, 4, 5, 6, 9}`, então os buckets 2, 3, 7 e 8 **nunca são usados**, e os buckets 1, 4, 6, 9 recebem o dobro da carga. A chance de duas chaves aleatórias colidirem sobe de 0,10 para 0,18. Nos dados: chaves aleatórias têm `χ²` 85 (contra 5 em `h₁`/`h₂`) e maior carga 26 (contra 13).
- **Pior caso comum às três:** múltiplos de 10 caem todos no bucket 0 (carga 100, `χ²` = 900), e a tabela vira uma lista de tamanho `n`, com acesso `O(n)`.

**Ranking de risco de colisão:** `h₃` > `h₁` = `h₂`. **Recomendação:** usar `M` primo e funções que misturem todos os bits da chave. A `HashTable` desta entrega usa capacidade prima por isso.

In [24]:
import random

M = 10
FUNCOES = {"h1 = k mod 10": lambda k: k % M,
           "h2 = 7k mod 10": lambda k: (7 * k) % M,
           "h3 = k² mod 10": lambda k: (k * k) % M}


def distribuicao(chaves, h):
    """Carga de cada um dos M buckets."""
    carga = [0] * M
    for k in chaves:
        carga[h(k)] += 1
    return carga


def chi2(carga):
    """Desvio da distribuição uniforme (0 = perfeita)."""
    esperado = sum(carga) / M
    return sum((c - esperado) ** 2 / esperado for c in carga)

In [25]:
print("Bucket de cada último dígito:")
for nome, h in FUNCOES.items():
    mapa = [h(d) for d in range(10)]
    print(f"  {nome:15} {mapa}  buckets alcançáveis: {len(set(mapa))}")
assert sorted(FUNCOES["h2 = 7k mod 10"](d) for d in range(10)) == list(range(10))   # h2 é permutação

rng = random.Random(12)
conjuntos = {"consecutivas 0..99": list(range(100)),
             "múltiplos de 10": [10 * i for i in range(100)],
             "pares": [2 * i for i in range(100)],
             "aleatórias": [rng.randrange(10 ** 6) for _ in range(100)]}

print(f"\n{'conjunto':>19} {'função':>8} {'usados':>6} {'maior carga':>11} {'χ²':>5}  carga por bucket")
for nome_c, chaves in conjuntos.items():
    for nome_f, h in FUNCOES.items():
        carga = distribuicao(chaves, h)
        usados = sum(1 for c in carga if c)
        print(f"{nome_c:>19} {nome_f[:2]:>8} {usados:>6} {max(carga):>11} {chi2(carga):>5.0f}  {carga}")

Bucket de cada último dígito:
  h1 = k mod 10   [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]  buckets alcançáveis: 10
  h2 = 7k mod 10  [0, 7, 4, 1, 8, 5, 2, 9, 6, 3]  buckets alcançáveis: 10
  h3 = k² mod 10  [0, 1, 4, 9, 6, 5, 6, 9, 4, 1]  buckets alcançáveis: 6

           conjunto   função usados maior carga    χ²  carga por bucket
 consecutivas 0..99       h1     10          10     0  [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
 consecutivas 0..99       h2     10          10     0  [10, 10, 10, 10, 10, 10, 10, 10, 10, 10]
 consecutivas 0..99       h3      6          20    80  [10, 20, 0, 0, 20, 10, 20, 0, 0, 20]
    múltiplos de 10       h1      1         100   900  [100, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    múltiplos de 10       h2      1         100   900  [100, 0, 0, 0, 0, 0, 0, 0, 0, 0]
    múltiplos de 10       h3      1         100   900  [100, 0, 0, 0, 0, 0, 0, 0, 0, 0]
              pares       h1      5          20   100  [20, 0, 20, 0, 20, 0, 20, 0, 20, 0]
              pares       h2      5  

## Resumo de complexidade

| Ex. | Algoritmo | Tempo | Evidência medida |
|---|---|---|---|
| 1 | laços aninhados → hash table | `Θ(N²)` → `O(N)` | `N² − 1` → `3N` |
| 2 | interseção com hash table | `O(N + M)` | ≤ `3N` acessos, 200.000 elementos |
| 3 | primeiro duplicado | `O(N)` | `2N + 1` acessos |
| 4 | letra ausente | `O(N)`, espaço `O(1)` | vetor de 26 posições |
| 5 | primeiro não repetido | `O(N)` | `3N` acessos |
| 6 | Selection Sort | `O(N²)` sempre | `N(N−1)/2` comparações em todo cenário |
| 7 | Insertion Sort | melhor `O(N)`, pior `O(N²)` | 999 × 499.500 comparações (N = 1000) |
| 9, 10 | `Stack`, `Queue` | `O(1)` por operação | overflow/underflow explícitos, FIFO validado |
| 11 | reversão com pilha | `O(N)` | rastro de push/pop |